# Customer Churn Baseline Notebook

This notebook uses the supplied `customer-churn-training.csv`. The dataset has only 12 rows, so results are illustrative and not production evidence.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix

df = pd.read_csv('customer-churn-training.csv')
df

## 1. Quality checks

Check missing values, duplicates, target balance, and basic ranges before modelling.

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget balance:\n', df['churned'].value_counts())
print('\nPlan type:\n', df['plan_type'].value_counts())
print('\nNumeric summary:\n', df.describe())

## 2. Non-ML baseline

Illustrative rule: flag churn risk when `last_login_days >= 10`. This is a transparent baseline, not a validated business policy.

In [ ]:
rule_pred = (df['last_login_days'] >= 10).astype(int)
print('Confusion matrix:\n', confusion_matrix(df['churned'], rule_pred))
print('Accuracy:', accuracy_score(df['churned'], rule_pred))
print('Precision:', precision_score(df['churned'], rule_pred, zero_division=0))
print('Recall:', recall_score(df['churned'], rule_pred, zero_division=0))
print('F1:', f1_score(df['churned'], rule_pred, zero_division=0))

## 3. Logistic-regression baseline

`customer_id` is excluded. Numeric features are standardized and `plan_type` is one-hot encoded. The split is stratified and uses a fixed random seed for reproducibility.

In [ ]:
X = df.drop(columns=['churned', 'customer_id'])
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

numeric_features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
categorical_features = ['plan_type']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

model = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results['actual'] = y_test.values
results['predicted'] = pred
results['churn_probability'] = prob
results

In [ ]:
print('Confusion matrix:\n', confusion_matrix(y_test, pred))
print('Accuracy:', accuracy_score(y_test, pred))
print('Precision:', precision_score(y_test, pred, zero_division=0))
print('Recall:', recall_score(y_test, pred, zero_division=0))
print('F1:', f1_score(y_test, pred, zero_division=0))
print('Balanced accuracy:', balanced_accuracy_score(y_test, pred))

print('\nIMPORTANT: The test set contains only 4 rows. Perfect scores on this split are not evidence that the model is production-ready.')

## 4. Interpretation and safeguards

- The dataset is too small for reliable generalization.
- `customer_id` is excluded from modelling.
- The prediction horizon and feature timestamps are not documented, so leakage cannot be ruled out.
- Model output should be used for human review, not automatic adverse action.
- The non-ML process should remain the fallback if the model is unavailable or fails agreed quality checks.